In [ ]:
import sys
from pathlib import Path
from typing import Literal

from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
#print(sys.executable)

/Library/Developer/CommandLineTools/usr/bin/python3


In [3]:
model = OpenAIChatModel(
    model_name="qwen3:4b-instruct",
    provider=OllamaProvider(
        base_url="http://localhost:11434/v1"
    ),
)

In [4]:
PROJECT_ROOT = Path.cwd()
REPOSITORY_ROOT = PROJECT_ROOT / "sample_repository"

print("Repository:", REPOSITORY_ROOT)
print("Exists:", REPOSITORY_ROOT.exists())

Repository: /Users/theo/Documents/Projects/programingProjects/agentic_AI_project/agentic-ai/sample_repository
Exists: True


In [5]:
class RepositoryFinding(BaseModel):
    file_path: str
    title: str
    severity: Literal["low", "medium", "high"]
    evidence: str
    explanation: str
    recommendation: str

In [6]:
review_agent = Agent(
    model,
    output_type=list[RepositoryFinding],
    model_settings={
        "temperature": 0.0,
    },
)

In [7]:
python_files = sorted(
    path.relative_to(REPOSITORY_ROOT).as_posix()
    for path in REPOSITORY_ROOT.rglob("*.py")
    if path.is_file()
)

print(python_files)

['app.py', 'test_user_service.py', 'user_service.py']


In [8]:
source_files: dict[str, str] = {}

for relative_path in python_files:
    file_path = REPOSITORY_ROOT / relative_path
    source_files[relative_path] = file_path.read_text(
        encoding="utf-8"
    )

for path, source in source_files.items():
    print(f"Loaded: {path} ({len(source)} characters)")

Loaded: app.py (201 characters)
Loaded: test_user_service.py (118 characters)
Loaded: user_service.py (130 characters)


In [9]:
source_bundle = "\n\n".join(
    f"===== {path} =====\n{source}"
    for path, source in source_files.items()
)

print(source_bundle)

===== app.py =====
from user_service import get_user_name


def main() -> None:
    user_id = input("Enter user ID: ")
    name = get_user_name(user_id)

    print(f"User: {name}")


if __name__ == "__main__":
    main()

===== test_user_service.py =====
from user_service import get_user_name


def test_get_existing_user() -> None:
    assert get_user_name("1") == "Theo"

===== user_service.py =====
USERS = {
    "1": "Theo",
    "2": "Alice",
    "3": "Bob",
}


def get_user_name(user_id: str) -> str:
    return USERS[user_id]


In [10]:
result = await review_agent.run(
    f"""
    Review the supplied Python repository.

    Only analyse the source code included below.

    Identify concrete software defects.

    Do not invent behaviour that is not present in the source.

    Every finding must:
    - identify the actual file containing the defect
    - quote or describe concrete evidence from that file
    - explain why the evidence represents a defect
    - recommend a specific correction

    Return an empty list if you cannot identify a concrete defect.

    Repository source:

    {source_bundle}
    """
)

In [11]:
print(type(result.output))

<class 'list'>


In [12]:
print(f"Number of findings: {len(result.output)}")

Number of findings: 1


In [15]:
for finding in result.output:
    print()
    print(f"File:     {finding.file_path}")
    print(f"Title:    {finding.title}")
    print(f"Severity: {finding.severity}")
    print()
    print("Evidence:")
    print(finding.evidence)
    print()
    print("Explanation:")
    print(finding.explanation)
    print()
    print("Recommendation:")
    print(finding.recommendation)
    print("=" * 80)


File:     user_service.py
Title:    Missing Error Handling in get_user_name Function
Severity: high

Evidence:
def get_user_name(user_id: str) -> str:\n    return USERS[user_id]

Explanation:
The function get_user_name directly accesses the USERS dictionary using the provided user_id as a key. If the user_id is not present in the USERS dictionary, this will raise a KeyError. There is no error handling or default value provided for missing user IDs.

Recommendation:
Modify the function to include a default value or error handling for missing user IDs. For example, add a condition to check if the user_id exists in the USERS dictionary before accessing it, and return a default value or raise a custom exception if it does not exist.


In [16]:
import subprocess


def run_repository_tests() -> str:
    result = subprocess.run(
        ["python", "-m", "pytest"],
        cwd=REPOSITORY_ROOT,
        capture_output=True,
        text=True,
    )

    return (
        f"Exit code: {result.returncode}\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

In [17]:
test_result = run_repository_tests()

print(test_result)

Exit code: 0

STDOUT:
============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0
rootdir: /Users/theo/Documents/Projects/programingProjects/agentic_AI_project/agentic-ai/sample_repository
plugins: langsmith-0.12.6, anyio-4.10.0
collected 1 item

test_user_service.py .                                                   [100%]

============================== 1 passed in 0.00s ===============================


STDERR:

